In [29]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import sys
sys.path.append("..")
from src.evaluacion.metricas import evaluar
from src.features.feature_sets import get_feature_sets, TARGET
from xgboost import XGBRegressor
from src.evaluacion.backtesting import walk_forward
import importlib
import src.evaluacion.metricas, src.evaluacion.backtesting
importlib.reload(src.evaluacion.metricas)
importlib.reload(src.evaluacion.backtesting)
from src.evaluacion.backtesting import walk_forward_cuantiles
from src.evaluacion.metricas import pinball_loss
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression




In [7]:
df =pd.read_parquet("../data/processed/tabla_features.parquet").sort_values("datetime_utc")
df = df[df["entrenable"]]

In [8]:
ultima_fecha = df["datetime_utc"].max()
corte = ultima_fecha - pd.DateOffset(months=6)
train = df[df["datetime_utc"] < corte] 
test = df[df["datetime_utc"] >= corte]
print(len(train))
print(len(test))

25871
4369


In [9]:

cuantiles = [0.05,0.1, 0.5, 0.9, 0.95]
feats = get_feature_sets(df)["predictivo"]
x_train, y_train = train[feats], train[TARGET]
modelos = {}
for q in cuantiles:
    modelo = XGBRegressor(
        objective="reg:quantileerror",
        quantile_alpha=q,
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        )
    modelo.fit(x_train, y_train)
    modelos[q] = modelo


In [10]:
X_test, y_test = test[feats], test[TARGET]

preds = pd.DataFrame(
    {q: modelos[q].predict(X_test) for q in cuantiles},
    index=test.index,
)
preds.head()

,0.05,0.10,0.50,0.90,0.95
26183,101.043724,104.458755,109.414948,125.916977,126.874123
26184,99.308784,104.861145,110.249809,116.466133,117.760719
26185,88.622566,98.150612,102.890579,113.228592,112.318581
26186,86.296074,96.238373,99.897888,111.022827,109.132195
26187,83.317230,95.918816,95.646667,106.073647,104.495262


In [14]:
preds_mono = pd.DataFrame(
    np.sort(preds.values, axis=1),   # ordena cada fila ascendente
    columns=preds.columns,
    index=preds.index,
)


In [12]:
n_cruces = (preds.values != np.sort(preds.values, axis=1)).any(axis=1).sum()
print(f"filas con cruce: {n_cruces} de {len(preds)} ({100*n_cruces/len(preds):.1f}%)")

filas con cruce: 1503 de 4369 (34.4%)


In [15]:
cob_80 = ((y_test >= preds_mono[0.10]) & (y_test <= preds_mono[0.90])).mean()
cob_90 = ((y_test >= preds_mono[0.05]) & (y_test <= preds_mono[0.95])).mean()
print(f"cobertura [P10, P90]: {cob_80:.3f}  (objetivo 0.80)")
print(f"cobertura [P05, P95]: {cob_90:.3f}  (objetivo 0.90)")

cobertura [P10, P90]: 0.500  (objetivo 0.80)
cobertura [P05, P95]: 0.684  (objetivo 0.90)


In [ ]:
por_encima = (y_test > preds_mono[0.90]).mean()   
por_debajo = (y_test < preds_mono[0.10]).mean()   
print(f"por encima de P90: {por_encima:.3f}")
print(f"por debajo de P10: {por_debajo:.3f}")
print("fracción por debajo de P50:", (y_test < preds_mono[0.5]).mean())  

por encima de P90: 0.139
por debajo de P10: 0.361
fracción por debajo de P50: 0.598992904554818


In [22]:
df["meses"] = df["datetime_utc"].dt.to_period("M")
def predecir_cuantiles(tr, te):
    ms = {}
    for q in cuantiles:
        m = XGBRegressor(objective="reg:quantileerror", quantile_alpha=q,
                         n_estimators=400, learning_rate=0.05, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8,
                         tree_method="hist", random_state=42, n_jobs=-1)
        m.fit(tr[feats], tr[TARGET])
        ms[q] = m
    p = pd.DataFrame({q: ms[q].predict(te[feats]) for q in cuantiles}, index=te.index)
    # monotoniza cada fila (rearrangement) para evitar cruces
    return pd.DataFrame(np.sort(p.values, axis=1), columns=p.columns, index=p.index)

df_cal = walk_forward_cuantiles(df, TARGET, cuantiles, predecir_cuantiles)
print(df_cal[["cob_80", "cob_90"]].mean())
df_cal

C:\Users\jaime\AppData\Local\Temp\ipykernel_10612\3009442260.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["meses"] = df["datetime_utc"].dt.to_period("M")


cob_80    0.511444
cob_90    0.669862
dtype: float64


,mes,cob_80,cob_90,pinball
0,2024-01,0.479839,0.604839,4.415316
1,2024-02,0.255747,0.422414,5.617864
2,2024-03,0.087366,0.211022,8.494520
3,2024-04,0.225000,0.311111,5.908329
4,2024-05,0.502688,0.670699,4.744731
5,2024-06,0.595833,0.740278,4.162762
6,2024-07,0.604839,0.802419,3.308858
7,2024-08,0.526882,0.709677,3.787660
8,2024-09,0.584722,0.683333,3.746570
9,2024-10,0.618280,0.780914,3.624901


In [23]:
# ── Conformalized Quantile Regression (CQR) ──────────────────────────────
# Recalibra las bandas usando un set que el modelo NO vio, para que la
# cobertura real se ajuste al objetivo (80% / 90%).

# 1) Set de CALIBRACIÓN = últimos 3 meses de train (lo más cercano al test)
corte_calib = train["datetime_utc"].max() - pd.DateOffset(months=3)
proper = train[train["datetime_utc"] < corte_calib]
calib  = train[train["datetime_utc"] >= corte_calib]
print(f"proper {len(proper)} | calib {len(calib)} | test {len(test)}")

# 2) Reentrenar los cuantiles SOLO con proper (calib queda fuera del entrenamiento)
def entrena_cuantiles(tr):
    ms = {}
    for q in cuantiles:
        m = XGBRegressor(objective="reg:quantileerror", quantile_alpha=q,
                         n_estimators=400, learning_rate=0.05, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8,
                         tree_method="hist", random_state=42, n_jobs=-1)
        m.fit(tr[feats], tr[TARGET])
        ms[q] = m
    return ms

def predice(ms, X):
    p = pd.DataFrame({q: ms[q].predict(X[feats]) for q in cuantiles}, index=X.index)
    return pd.DataFrame(np.sort(p.values, axis=1), columns=p.columns, index=p.index)  # monotoniza

modelos_cqr = entrena_cuantiles(proper)
pred_calib = predice(modelos_cqr, calib)
pred_test  = predice(modelos_cqr, test)

# 3) Corrección conforme por banda: cuánto hay que ensanchar
def correccion(y, lo, hi, alpha):
    e = np.maximum(lo - y, y - hi)          # score: cuánto se sale el real (negativo si dentro)
    n = len(e)
    nivel = np.ceil((n + 1) * (1 - alpha)) / n   # cuantil conforme (corrección muestra finita)
    return np.quantile(e, min(nivel, 1.0))

y_calib = calib[TARGET].to_numpy()
Q80 = correccion(y_calib, pred_calib[0.10].to_numpy(), pred_calib[0.90].to_numpy(), 0.20)
Q90 = correccion(y_calib, pred_calib[0.05].to_numpy(), pred_calib[0.95].to_numpy(), 0.10)
print(f"ensanche Q80: {Q80:.2f} €/MWh | Q90: {Q90:.2f} €/MWh")

# 4) Aplicar al test y comparar cobertura antes/después
y_test = test[TARGET].to_numpy()
cob = lambda y, lo, hi: ((y >= lo) & (y <= hi)).mean()

print("\n           antes → después   (objetivo)")
print(f"cob_80   {cob(y_test, pred_test[0.10].to_numpy(), pred_test[0.90].to_numpy()):.3f} → "
      f"{cob(y_test, pred_test[0.10].to_numpy()-Q80, pred_test[0.90].to_numpy()+Q80):.3f}   (0.80)")
print(f"cob_90   {cob(y_test, pred_test[0.05].to_numpy(), pred_test[0.95].to_numpy()):.3f} → "
      f"{cob(y_test, pred_test[0.05].to_numpy()-Q90, pred_test[0.95].to_numpy()+Q90):.3f}   (0.90)")


proper 23830 | calib 2041 | test 4369
ensanche Q80: 4.18 €/MWh | Q90: 4.27 €/MWh

           antes → después   (objetivo)
cob_80   0.511 → 0.724   (0.80)
cob_90   0.680 → 0.863   (0.90)


In [24]:
# ── CQR asimétrico: corrección independiente por cada lado ──
def q_conf(scores, nivel_obj):
    n = len(scores)
    nivel = np.ceil((n + 1) * nivel_obj) / n
    return np.quantile(scores, min(nivel, 1.0))

def cqr_asimetrico(lo_col, hi_col, alpha):
    # repartimos alpha entre las dos colas: alpha/2 cada una
    r_lo = pred_calib[lo_col].to_numpy() - y_calib   # >0 si el real cae por debajo del límite bajo
    r_hi = y_calib - pred_calib[hi_col].to_numpy()   # >0 si el real cae por encima del límite alto
    c_lo = q_conf(r_lo, 1 - alpha/2)
    c_hi = q_conf(r_hi, 1 - alpha/2)
    low  = pred_test[lo_col].to_numpy() - c_lo
    high = pred_test[hi_col].to_numpy() + c_hi
    return low, high, c_lo, c_hi

cob = lambda y, lo, hi: ((y >= lo) & (y <= hi)).mean()

low80, high80, clo80, chi80 = cqr_asimetrico(0.10, 0.90, 0.20)
low90, high90, clo90, chi90 = cqr_asimetrico(0.05, 0.95, 0.10)

print(f"80%: ensanche abajo {clo80:.2f} | arriba {chi80:.2f}  → cobertura {cob(y_test, low80, high80):.3f} (obj 0.80)")
print(f"90%: ensanche abajo {clo90:.2f} | arriba {chi90:.2f}  → cobertura {cob(y_test, low90, high90):.3f} (obj 0.90)")

80%: ensanche abajo 6.25 | arriba 0.74  → cobertura 0.736 (obj 0.80)
90%: ensanche abajo 7.53 | arriba 0.22  → cobertura 0.880 (obj 0.90)


In [25]:
for umbral, nombre in [(0, "precio < 0"), (150, "precio > 150")]:
    if umbral == 0:
        freq_train = (train[TARGET] < 0).mean()
        freq_test  = (test[TARGET]  < 0).mean()
    else:
        freq_train = (train[TARGET] > 150).mean()
        freq_test  = (test[TARGET]  > 150).mean()
    print(f"{nombre}:  train {freq_train:.1%}  |  test {freq_test:.1%}")

precio < 0:  train 1.5%  |  test 11.3%
precio > 150:  train 3.0%  |  test 1.1%


In [27]:


y_train_neg = (train[TARGET] < 0).astype(int)
y_test_neg  = (test[TARGET]  < 0).astype(int)

clf_neg = XGBClassifier(
    objective="binary:logistic",
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", random_state=42, n_jobs=-1,
)
clf_neg.fit(train[feats], y_train_neg)
p_neg = clf_neg.predict_proba(test[feats])[:, 1]   # prob. de precio < 0

print("DISCRIMINACIÓN (¿acierta QUÉ horas?)")
print(f"  ROC-AUC: {roc_auc_score(y_test_neg, p_neg):.3f}  (0.5 = azar)")
print(f"  PR-AUC : {average_precision_score(y_test_neg, p_neg):.3f}  (base rate = {y_test_neg.mean():.3f})")
print("CALIBRACIÓN (¿son correctas las probabilidades?)")
print(f"  prob media predicha: {p_neg.mean():.3f}  vs  frecuencia real: {y_test_neg.mean():.3f}")
print(f"  Brier: {brier_score_loss(y_test_neg, p_neg):.4f}")

DISCRIMINACIÓN (¿acierta QUÉ horas?)
  ROC-AUC: 0.935  (0.5 = azar)
  PR-AUC : 0.666  (base rate = 0.113)
CALIBRACIÓN (¿son correctas las probabilidades?)
  prob media predicha: 0.013  vs  frecuencia real: 0.113
  Brier: 0.0978


In [ ]:

p_neg = clf_neg.predict_proba(test[feats])[:, 1]
mitad = len(test) // 2

# aprende prob_cruda -> frecuencia real en el 1er tramo del test
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(p_neg[:mitad], y_test_neg.values[:mitad])
p_recal = iso.predict(p_neg[mitad:])          # aplica al 2º tramo

y_eval = y_test_neg.values[mitad:]
p_crudo = p_neg[mitad:]
print(f"frecuencia real:        {y_eval.mean():.3f}")
print(f"prob media CRUDA:       {p_crudo.mean():.3f}")
print(f"prob media RECALIBRADA: {p_recal.mean():.3f}")
print(f"AUC crudo {roc_auc_score(y_eval,p_crudo):.3f} → recalibrado {roc_auc_score(y_eval,p_recal):.3f}")
print(f"Brier crudo {brier_score_loss(y_eval,p_crudo):.4f} → recalibrado {brier_score_loss(y_eval,p_recal):.4f}")

frecuencia real:        0.122
prob media CRUDA:       0.024
prob media RECALIBRADA: 0.222
AUC crudo 0.957 → recalibrado 0.955
Brier crudo 0.0931 → recalibrado 0.0806
